In [21]:
import os
import pandas as pd

# [1] Cấu hình thư mục gốc
DATA_DIR = "/Users/nguyenminhtri/FinalYearPro/data/raw"

# Nhóm Bảng chính (Main Tables)
MAIN_TRAIN_FILE = os.path.join(DATA_DIR, "application_train.csv")
MAIN_TEST_FILE = os.path.join(DATA_DIR, "application_test.csv")

# Nhóm Luồng lịch sử BÊN NGOÀI Home Credit (Credit Bureau Stream)
BUREAU_FILE = os.path.join(DATA_DIR, "bureau.csv")
BUREAU_BAL_FILE = os.path.join(DATA_DIR, "bureau_balance.csv")

# Nhóm Luồng lịch sử BÊN TRONG Home Credit (Internal Behavioral Stream)
PREV_APP_FILE = os.path.join(DATA_DIR, "previous_application.csv")
INS_PAYMENT_FILE = os.path.join(DATA_DIR, "installments_payments.csv")
POS_CASH_FILE = os.path.join(DATA_DIR, "POS_CASH_balance.csv")
CREDIT_CARD_FILE = os.path.join(DATA_DIR, "credit_card_balance.csv")

# File phụ bổ sung thông tin mô tả
COL_DESC_FILE = os.path.join(DATA_DIR, "HomeCredit_columns_description.csv")

In [22]:
import json
metadata = {
    "BUREAU_BALANCE": {
        "group_col": "SK_ID_BUREAU",
        "numeric_agg": {"MONTHS_BALANCE": ["min", "max", "count"]},
        "categorical_cols": ["STATUS"]
    }
}

# 2. Load dữ liệu thử nghiệm
df = pd.read_csv(BUREAU_BAL_FILE, nrows=10000)
print("Load dữ liệu thành công, shape:", df.shape)

Load dữ liệu thành công, shape: (10000, 3)


In [23]:
import pandas as pd

# 1. Giả sử df là bảng bureau_balance của bạn
# df = pd.read_csv('.../bureau_balance.csv')

# 2. Gộp các trạng thái nợ xấu (1, 2, 3, 4, 5) thành 'BAD_DEBT'
# Những trạng thái 'C', '0', 'X' giữ nguyên
status_mapping = {
    '1': 'BAD_DEBT',
    '2': 'BAD_DEBT',
    '3': 'BAD_DEBT',
    '4': 'BAD_DEBT',
    '5': 'BAD_DEBT'
}

# Thay thế giá trị trong cột STATUS
df['STATUS_CLEAN'] = df['STATUS'].replace(status_mapping)

# 3. Chuyển thành dạng One-Hot (tự động tạo cột STATUS_CLEAN_C, STATUS_CLEAN_0, STATUS_CLEAN_X, STATUS_CLEAN_BAD_DEBT)
df_encoded = pd.get_dummies(df, columns=['STATUS_CLEAN'])

# 4. Gom nhóm theo ID và đếm tổng số lần xuất hiện của từng trạng thái (tính tổng)
# Đây là bước "Đếm" mà bạn mong muốn
agg_df = df_encoded.groupby('SK_ID_BUREAU').sum(numeric_only=True)

# 5. Kiểm tra kết quả
print("Các cột mới đã tạo ra:")
print(agg_df.columns.tolist())
print("\nKết quả mẫu:")
print(agg_df.head())

Các cột mới đã tạo ra:
['MONTHS_BALANCE', 'STATUS_CLEAN_0', 'STATUS_CLEAN_BAD_DEBT', 'STATUS_CLEAN_C', 'STATUS_CLEAN_X']

Kết quả mẫu:
              MONTHS_BALANCE  STATUS_CLEAN_0  STATUS_CLEAN_BAD_DEBT  \
SK_ID_BUREAU                                                          
5715448                 -351               8                      0   
5715449                  -66               5                      0   
5715451                 -455              17                      0   
5715452                 -528               8                      0   
5715453                 -703               8                      0   

              STATUS_CLEAN_C  STATUS_CLEAN_X  
SK_ID_BUREAU                                  
5715448                    9              10  
5715449                    6               1  
5715451                    5               4  
5715452                   15              10  
5715453                   20              10  


In [24]:
def validate_metadata(df, table_name, meta):
    """
    Kiểm tra xem các yêu cầu trong Metadata có khớp với DataFrame thực tế không.
    """
    # 1. Kiểm tra cột ID có tồn tại không
    if meta['group_col'] not in df.columns:
        raise ValueError(f"LỖI: Bảng {table_name} thiếu cột {meta['group_col']}!")

    # 2. Kiểm tra các cột trong 'numeric' có thực sự tồn tại trong file không
    if 'numeric' in meta['aggregations']:
        for col in meta['aggregations']['numeric'].keys():
            if col not in df.columns:
                raise ValueError(f"LỖI: Bảng {table_name} không có cột {col} để tính toán!")

    # 3. Kiểm tra các cột 'categorical'
    if 'categorical' in meta['aggregations']:
        for col in meta['categorical'].keys():
            if col not in df.columns:
                raise ValueError(f"LỖI: Bảng {table_name} không có cột {col} để phân loại!")

    print(f"✅ Metadata của {table_name} hợp lệ!")
    return True

In [25]:

def prototype_process(df, table_name, meta):
    # Gom nhóm số
    agg = df.groupby(meta['group_col']).agg(meta['numeric_agg'])
    agg.columns = [f"{table_name}_{col[0]}_{col[1]}" for col in agg.columns]

    # Gom nhóm phân loại (Dummies)
    for col in meta['categorical_cols']:
        dummies = pd.get_dummies(df[[meta['group_col'], col]], columns=[col])
        dummies = dummies.groupby(meta['group_col']).sum()
        # Đặt tên prefix cho dummies
        dummies.columns = [f"{table_name}_{c}" for c in dummies.columns]
        agg = agg.join(dummies, on=meta['group_col'])

    return agg

# Chạy thử
processed_df = prototype_process(df, "BUREAU_BAL", metadata["BUREAU_BALANCE"])
processed_df.head()

,BUREAU_BAL_MONTHS_BALANCE_min,BUREAU_BAL_MONTHS_BALANCE_max,BUREAU_BAL_MONTHS_BALANCE_count,BUREAU_BAL_STATUS_0,BUREAU_BAL_STATUS_1,BUREAU_BAL_STATUS_2,BUREAU_BAL_STATUS_3,BUREAU_BAL_STATUS_C,BUREAU_BAL_STATUS_X
SK_ID_BUREAU,,,,,,,,,
5715448,-26,0,27,8,0,0,0,9,10
5715449,-11,0,12,5,0,0,0,6,1
5715451,-30,-5,26,17,0,0,0,5,4
5715452,-32,0,33,8,0,0,0,15,10
5715453,-37,0,38,8,0,0,0,20,10


In [26]:
# Xem qua dữ liệu kết quả
print("Kết quả sau khi xử lý:")
display(processed_df.describe())

# Check xem có NaN không
print("\nSố lượng NaN:")
print(processed_df.isna().sum().sum())

Kết quả sau khi xử lý:


,BUREAU_BAL_MONTHS_BALANCE_min,BUREAU_BAL_MONTHS_BALANCE_max,BUREAU_BAL_MONTHS_BALANCE_count,BUREAU_BAL_STATUS_0,BUREAU_BAL_STATUS_1,BUREAU_BAL_STATUS_2,BUREAU_BAL_STATUS_3,BUREAU_BAL_STATUS_C,BUREAU_BAL_STATUS_X
count,273.000000,273.000000,273.000000,273.000000,273.000000,273.000000,273.000000,273.000000,273.000000
mean,-38.109890,-2.479853,36.630037,9.109890,0.249084,0.029304,0.007326,18.344322,8.890110
std,25.869917,11.265535,24.934593,9.544151,0.998275,0.224964,0.085435,24.627343,16.034281
min,-96.000000,-90.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-53.000000,0.000000,18.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-32.000000,0.000000,30.000000,7.000000,0.000000,0.000000,0.000000,7.000000,1.000000
75%,-19.000000,0.000000,50.000000,12.000000,0.000000,0.000000,0.000000,31.000000,10.000000
max,0.000000,0.000000,97.000000,50.000000,10.000000,2.000000,1.000000,97.000000,94.000000



Số lượng NaN:
0
